In [ ]:
import pdfplumber
import pandas as pd
from datetime import datetime
import re

electric_details = {}
gas_details = {}

pdf_path = "BGE/20250716.pdf"
# pdf_path = "BGE/20250815.pdf"
# pdf_path = "BGE/20250616.pdf"

# Function returns standard date output; Jun22,2025 -> 2025-06-22
def format_date(input_date:str):
    # Expects abbreviated month name
    date_obj = datetime.strptime(input_date, "%b%d,%Y")
    format_date = date_obj.strftime("%Y-%m-%d")
    return format_date

def initialize_dict(output_dict:dict, filter:str):
    if filter == "ELECTRIC":
        output_dict["total_kWh"] = 0
        output_dict["total_price_electric"] = 0
        output_dict["electric_supply_0_rate"] = 0
        output_dict["electric_supply_0_energy"] = 0
        output_dict["electric_supply_0_price"] = 0
        output_dict["delivery_cust_price"] = 0
        output_dict["delivery_empower_md_rate"] = 0
        output_dict["deliver_empower_md_price"] = 0
        output_dict["md_svc_prog_fee_price"] = 0
        output_dict["env_surchg_fee_rate"] = 0
        output_dict["env_surchg_fee_price"] = 0
        output_dict["franchise_tax_rate"] = 0
        output_dict["franchise_tax_price"] = 0

def extract_electric(input_table:list, output_dict:dict):
    # Initialize dictionary
    initialize_dict("ELECTRIC", output_dict)
    for item in input_table:
        # Filter out NoneType entries
        row = item
        row = (item[0] or "").splitlines()
        for entry in row:
            # Total Energy
            if "Current - Previous" in entry:
                output_dict["total_kWh"] = float(entry.split("= ")[1])
            
            # Total Price
            if "TOTAL" in entry:
                output_dict["total_price_electric"] = float(re.search(r"\$(\d+\.\d+)", entry).group(1))
            # Billing Periods
            if "BillingPeriod" in entry:
                date_paentryern = r'[A-Z][a-z]{2}\d{1,2},\d{4}'
                dates = re.findall(date_paentryern, entry)
                output_dict["billing_period_start"] = format_date(dates[0])
                output_dict["billing_period_end"] = format_date(dates[1])

            # Electric Supply
            if "ELECTRICSUPPLY" in entry:
                indices = [i for i, target in enumerate(row) if "ELECTRICSUPPLY" in target or "BGEELECTRICDELIVERY" in target]
                number_rates = indices[1] - indices[0] - 1
                for i in range(0,number_rates):
                    # Skip first entry b/c it's an anchor
                    search_idx = row[i+1]
                    rate_key = f"electric_supply_{i}_rate"
                    rate_energy_key = f"electric_supply_{i}_energy"
                    rate_price_key = f"electric_supply_{i}_price"

                    # Skips first entry
                    output_dict[rate_energy_key] = float(re.search(r"(\d+(?:\.\d+)?)kWh", search_idx).group(1))
                    output_dict[rate_key] = float(re.search(r"x\s*(\.\d+)", search_idx).group(1))
                    output_dict[rate_price_key] = search_idx.split(" ")[-1]
            
            # BGE Electric Delivery - assumes all rates agai
            ## Customer Charge
            if "CustomerCharge" in entry:
                output_dict["delivery_cust_price"] = entry.split(" ")[1]
            ## EmPower MD Charge
            if "EmPowerMDChg" in entry:
                output_dict["delivery_empower_md_rate"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))
                output_dict["deliver_empower_md_price"] = entry.split(" ")[-1]
            ## Distribution Charge
            if "DistributionChg" in entry:
                output_dict["delivery_distribution_rate"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))
                output_dict["delivery_distribution_price"] = entry.split(" ")[-1]

            # Taxes and Fees
            ## Maryland Universal Service Program
            if "MDUniversalSvcProg" in entry:
                output_dict["md_svc_prog_fee_price"] = entry.split(" ")[1]
            ## Environmental Surcharge
            if "EnvirSrchg" in entry:
                output_dict["env_surchg_fee_rate"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))
                output_dict["env_surchg_fee_price"] = entry.split(" ")[-1]
            ## Franchise Tax
            if "FranchiseTax" in entry:
                output_dict["franchise_tax_rate"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))
                output_dict["franchise_tax_price"] = entry.split(" ")[-1]


with pdfplumber.open(pdf_path) as pdf:
    first_page = pdf.pages[1]
    table = first_page.extract_tables()
    electric_table = table[0]
    gas_table = table[1]
    extract_electric(electric_table, electric_details)
    print(electric_details)
    # df = pd.DataFrame(table[1:], columns=table[0]) 



{'billing_period_start': '2025-05-22', 'billing_period_end': '2025-06-23', 'total_kWh': 751.0, 'electric_supply_0_energy': 211.22, 'electric_supply_0_rate': 0.11899, 'electric_supply_0_price': '25.13', 'electric_supply_1_energy': 539.78, 'electric_supply_1_rate': 0.12587, 'electric_supply_1_price': '67.94', 'delivery_cust_price': '9.65', 'delivery_empower_md_rate': 0.01028, 'deliver_empower_md_price': '7.72', 'delivery_distribution_rate': 0.0436, 'delivery_distribution_price': '32.74', 'md_svc_prog_fee_price': '0.32', 'env_surchg_fee_rate': 0.00015, 'env_surchg_fee_price': '0.11', 'franchise_tax_rate': 0.00062, 'franchise_tax_price': '0.47', 'total_price_electric': 144.08}
